In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

# Configuration
MODELS = {
    "Mistral Medium": {
        "contamination_dir": Path("phase3_bcq/mistral-medium-latest"),
        "performance_dir": Path("../v3/mistral_medium"),
    },
    "GPT-5.2": {
        "contamination_dir": Path("phase3_bcq/gpt-5.2-2025-12-11"),
        "performance_dir": Path("../v3/gpt_5_2_openai"),
    },
}

MODEL_COLORS = {"Mistral Medium": "#e74c3c", "GPT-5.2": "#3498db"}
PERFORMANCE_CONFIG = "c0"

# Domain categories
DOMAINS = {
    "Scientific": ["ABSTRCT", "SCIARK", "ARGUMINSCI"],
    "Legal/Finance": ["ACQUA", "FINARG"],
    "Essays": ["AEC", "AFS", "PE"],
    "Discourse": ["IAM", "USELEC"],
}
DOMAIN_MARKERS = {"Scientific": "o", "Legal/Finance": "s", "Essays": "^", "Discourse": "D"}

def get_domain(dataset):
    for domain, datasets in DOMAINS.items():
        if dataset in datasets:
            return domain
    return "Other"

In [2]:
def load_contamination(model_dir: Path) -> pd.DataFrame:
    """Load all contamination reports for a model."""
    records = []
    for f in model_dir.glob("*_contamination_report.json"):
        if f.name.startswith("dep_"):  # skip deprecated
            continue
        with open(f) as fp:
            data = json.load(fp)
            records.append({
                "dataset": data["dataset"],
                "min_contamination": data["min_contamination"],
                "max_contamination": data["max_contamination"],
            })
    return pd.DataFrame(records).sort_values("dataset").reset_index(drop=True)


def load_performance(perf_dir: Path, config_prefix: str) -> pd.DataFrame:
    """Load performance (macro-F1) from v3 results."""
    records = []
    for f in perf_dir.glob(f"{config_prefix}_*.json"):
        with open(f) as fp:
            data = json.load(fp)
            for ds, metrics in data["datasets"].items():
                f1 = metrics["reports"]["original"]["macro avg"]["f1-score"]
                records.append({"dataset": ds, "macro_f1": f1 * 100})
    return pd.DataFrame(records).sort_values("dataset").reset_index(drop=True)

In [3]:
# Load data for all models
all_data = []
for model_name, paths in MODELS.items():
    cont_df = load_contamination(paths["contamination_dir"])
    perf_df = load_performance(paths["performance_dir"], PERFORMANCE_CONFIG)
    merged = cont_df.merge(perf_df, on="dataset")
    merged["model"] = model_name
    all_data.append(merged)

df = pd.concat(all_data, ignore_index=True)
df

,dataset,min_contamination,max_contamination,macro_f1,model
0,ABSTRCT,68.0,68.0,76.430976,mistral-medium-latest
1,ACQUA,58.0,58.0,73.214286,mistral-medium-latest
2,AEC,36.0,36.0,44.444444,mistral-medium-latest
3,AFS,34.0,34.0,45.054945,mistral-medium-latest
4,ARGUMINSCI,52.0,52.0,57.637997,mistral-medium-latest
5,FINARG,42.0,42.0,36.507937,mistral-medium-latest
6,IAM,12.0,12.0,66.063348,mistral-medium-latest
7,PE,46.0,46.0,64.114833,mistral-medium-latest
8,SCIARK,68.0,68.0,86.666667,mistral-medium-latest
9,USELEC,46.0,46.0,56.228956,mistral-medium-latest


In [ ]:
# Plot 1: Contamination comparison - Both models side by side
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(12, 5))

datasets = df["dataset"].unique()
x = np.arange(len(datasets))
width = 0.35

for i, model in enumerate(df["model"].unique()):
    model_df = df[df["model"] == model].set_index("dataset").loc[datasets]
    
    # Bar with error range (min to max)
    bars = ax.bar(x + i * width - width/2, model_df["max_contamination"], width,
                  label=model, color=MODEL_COLORS[model], alpha=0.85, edgecolor="white", linewidth=1)
    
    # Error bars showing range
    yerr_low = model_df["max_contamination"] - model_df["min_contamination"]
    ax.errorbar(x + i * width - width/2, model_df["max_contamination"], 
                yerr=[yerr_low, [0]*len(yerr_low)], fmt="none", color="black", capsize=3, alpha=0.6)

ax.axhline(y=20, color="gray", linestyle="--", alpha=0.7, label="Random (20%)")
ax.set_xticks(x)
ax.set_xticklabels(datasets, rotation=45, ha="right", fontsize=10)
ax.set_ylabel("Contamination (%)", fontsize=11)
ax.set_xlabel("Dataset", fontsize=11)
ax.set_title("DCQ Contamination Levels by Model and Dataset", fontsize=13, fontweight="bold")
ax.legend(loc="upper right")
ax.set_ylim(0, 80)

plt.tight_layout()
plt.savefig("contamination_comparison.pdf", dpi=300, bbox_inches="tight")
plt.savefig("contamination_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

# Summary stats
print("\nContamination Summary:")
print("-" * 60)
for model in df["model"].unique():
    model_df = df[df["model"] == model]
    print(f"{model}: mean={model_df['max_contamination'].mean():.1f}%, "
          f"range=[{model_df['max_contamination'].min():.0f}%, {model_df['max_contamination'].max():.0f}%]")

In [ ]:
# Plot 2: Contamination vs Performance - Individual models (side by side)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

correlations = {}
for idx, model in enumerate(df["model"].unique()):
    ax = axes[idx]
    model_df = df[df["model"] == model].copy()
    model_df["domain"] = model_df["dataset"].apply(get_domain)
    
    x = model_df["max_contamination"].values
    y = model_df["macro_f1"].values
    
    # Regression
    slope, intercept, r, p, _ = stats.linregress(x, y)
    correlations[model] = {"r": r, "p": p, "r_sq": r**2, "slope": slope}
    
    x_line = np.linspace(0, 80, 100)
    ax.plot(x_line, slope * x_line + intercept, color=MODEL_COLORS[model], lw=2, alpha=0.7, linestyle="--")
    
    # Scatter by domain
    for domain, marker in DOMAIN_MARKERS.items():
        domain_df = model_df[model_df["domain"] == domain]
        if len(domain_df) > 0:
            ax.scatter(domain_df["max_contamination"], domain_df["macro_f1"],
                      s=100, c=MODEL_COLORS[model], marker=marker, label=domain,
                      edgecolor="white", linewidth=1.5, zorder=5)
    
    # Labels
    for _, row in model_df.iterrows():
        ax.annotate(row["dataset"], (row["max_contamination"], row["macro_f1"]),
                   fontsize=8, alpha=0.7, xytext=(5, 5), textcoords="offset points")
    
    ax.set_xlabel("Contamination (%)", fontsize=11)
    ax.set_title(f"{model}\nr = {r:.2f}, p = {p:.3f}", fontsize=12, fontweight="bold")
    ax.set_xlim(0, 75)
    ax.set_ylim(30, 100)
    if idx == 0:
        ax.set_ylabel("Macro F1 (%)", fontsize=11)
        ax.legend(title="Domain", loc="lower right", fontsize=9)

plt.suptitle("Contamination vs Performance by Model", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("contamination_vs_performance_split.pdf", dpi=300, bbox_inches="tight")
plt.savefig("contamination_vs_performance_split.png", dpi=300, bbox_inches="tight")
plt.show()

# Print correlation table
print("\nCorrelation Summary:")
print("-" * 55)
print(f"{'Model':<20} {'r':>8} {'p-value':>10} {'R²':>8} {'Slope':>8}")
print("-" * 55)
for model, vals in correlations.items():
    sig = "**" if vals["p"] < 0.01 else "*" if vals["p"] < 0.05 else ""
    print(f"{model:<20} {vals['r']:>7.3f}{sig} {vals['p']:>10.4f} {vals['r_sq']:>7.1%} {vals['slope']:>8.2f}")
print("-" * 55)

In [ ]:
# Plot 3: Combined - Both models on same plot
fig, ax = plt.subplots(figsize=(10, 7))

for model in df["model"].unique():
    model_df = df[df["model"] == model].copy()
    model_df["domain"] = model_df["dataset"].apply(get_domain)
    
    x = model_df["max_contamination"].values
    y = model_df["macro_f1"].values
    
    # Regression line
    slope, intercept, r, p, _ = stats.linregress(x, y)
    x_line = np.linspace(0, 80, 100)
    ax.plot(x_line, slope * x_line + intercept, color=MODEL_COLORS[model], 
            lw=2.5, alpha=0.8, linestyle="--", label=f"{model} (r={r:.2f})")
    
    # Scatter
    ax.scatter(model_df["max_contamination"], model_df["macro_f1"],
              s=120, c=MODEL_COLORS[model], edgecolor="white", linewidth=1.5, zorder=5, alpha=0.85)

# Dataset labels (once, using average position)
for dataset in df["dataset"].unique():
    ds_df = df[df["dataset"] == dataset]
    avg_x = ds_df["max_contamination"].mean()
    avg_y = ds_df["macro_f1"].mean()
    ax.annotate(dataset, (avg_x, avg_y), fontsize=8, alpha=0.6, 
               xytext=(0, -15), textcoords="offset points", ha="center")

ax.axhline(y=50, color="gray", linestyle=":", alpha=0.4)
ax.axvline(x=20, color="gray", linestyle=":", alpha=0.4, label="Random chance")

ax.set_xlabel("DCQ Contamination (%)", fontsize=12)
ax.set_ylabel("Zero-Shot Macro F1 (%)", fontsize=12)
ax.set_title("Data Contamination Correlates with Performance\nAcross Both Models", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.set_xlim(0, 75)
ax.set_ylim(30, 100)

plt.tight_layout()
plt.savefig("contamination_vs_performance_combined.pdf", dpi=300, bbox_inches="tight")
plt.savefig("contamination_vs_performance_combined.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 4: Context effect on contamination-performance link (per model)
CONTEXT_LABELS = {"c0": "No context", "c1": "Definition", "c2": "+ Guideline", "c3": "+ Document"}

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True, sharex=True)

all_correlations = {}
for row_idx, (model_name, paths) in enumerate(MODELS.items()):
    all_correlations[model_name] = {}
    
    for col_idx, (config, label) in enumerate(CONTEXT_LABELS.items()):
        ax = axes[row_idx, col_idx]
        
        cont_df = load_contamination(paths["contamination_dir"])
        perf_df = load_performance(paths["performance_dir"], config)
        merged = cont_df.merge(perf_df, on="dataset")
        
        if len(merged) < 3:
            ax.set_visible(False)
            continue
        
        x, y = merged["max_contamination"].values, merged["macro_f1"].values
        slope, intercept, r, p, _ = stats.linregress(x, y)
        all_correlations[model_name][config] = {"r": r, "p": p}
        
        # Regression line
        x_line = np.linspace(0, 75, 50)
        ax.plot(x_line, slope * x_line + intercept, MODEL_COLORS[model_name], lw=2, alpha=0.7)
        ax.scatter(x, y, s=50, c=MODEL_COLORS[model_name], edgecolor="white", zorder=5)
        
        sig = "*" if p < 0.05 else ""
        ax.set_title(f"{label}\nr={r:.2f}{sig}", fontsize=10)
        ax.set_xlim(0, 75)
        ax.set_ylim(30, 100)
        
        if col_idx == 0:
            ax.set_ylabel(f"{model_name}\nMacro F1 (%)", fontsize=10)
        if row_idx == 1:
            ax.set_xlabel("Contamination (%)", fontsize=10)

plt.suptitle("Does Context Reduce the Contamination-Performance Link?", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("contamination_by_context_both_models.pdf", dpi=300, bbox_inches="tight")
plt.savefig("contamination_by_context_both_models.png", dpi=300, bbox_inches="tight")
plt.show()

# Summary table
print("\nCorrelation by Context Level:")
print("-" * 65)
print(f"{'Model':<20} {'c0':>10} {'c1':>10} {'c2':>10} {'c3':>10}")
print("-" * 65)
for model, configs in all_correlations.items():
    row = f"{model:<20}"
    for config in ["c0", "c1", "c2", "c3"]:
        if config in configs:
            sig = "*" if configs[config]["p"] < 0.05 else ""
            row += f" {configs[config]['r']:>8.2f}{sig}"
        else:
            row += f" {'N/A':>9}"
    print(row)
print("-" * 65)
print("* p < 0.05")

In [ ]:
# Final Summary Table for Thesis
print("=" * 70)
print("THESIS SUMMARY: Data Contamination Analysis")
print("=" * 70)

# Contamination levels
print("\n1. Contamination Levels (DCQ Max %):")
pivot = df.pivot(index="dataset", columns="model", values="max_contamination")
print(pivot.to_string())

print(f"\n   Mean contamination:")
for model in df["model"].unique():
    mean_cont = df[df["model"] == model]["max_contamination"].mean()
    print(f"   - {model}: {mean_cont:.1f}%")

# Correlation summary
print("\n2. Contamination-Performance Correlation (c0 baseline):")
for model in df["model"].unique():
    model_df = df[df["model"] == model]
    r, p = stats.pearsonr(model_df["max_contamination"], model_df["macro_f1"])
    print(f"   - {model}: r = {r:.3f}, p = {p:.4f}, R² = {r**2:.1%}")

# Key finding
print("\n" + "=" * 70)
print("KEY FINDING: Strong positive correlation between contamination and")
print("performance suggests benchmark leakage inflates reported accuracy.")
print("=" * 70)